In [ ]:
# ============================================================
# Cell 1 — Install dependencies
# ============================================================
!pip -q install -U google-genai langgraph pymupdf4llm pymupdf pydantic phonenumbers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

In [ ]:
# ============================================================
# Cell 2 — Imports and configuration
# ============================================================
import os
import json
import re
from pathlib import Path
from typing import TypedDict, Optional

from pydantic import BaseModel, Field, ValidationError
from google import genai
from langgraph.graph import StateGraph, START, END

import fitz  # PyMuPDF fallback
import pymupdf4llm

from google.colab import files

from google.colab import userdata


API_KEY = userdata.get('GEMINI_API_KEY')
if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set in the Colab environment.")

# Use an env override so you can switch models without changing the code.
# Current docs expose flash-family targets such as gemini-3-flash-preview and gemini-3.1-flash-lite-preview.
MODEL_NAME = os.getenv("GEMINI_MODEL_NAME", "gemini-2.5-flash")

client = genai.Client(api_key=API_KEY)

print("Model:", MODEL_NAME)

Model: gemini-2.5-flash


In [ ]:
# ============================================================
# Cell 3 — Pydantic schema for resume extraction
# ============================================================
from pydantic import BaseModel, Field
from typing import Optional


class ContactInfo(BaseModel):
    email: Optional[str] = Field(
        default=None, description="Primary email address.")
    phone_raw: Optional[str] = Field(
        default=None, description="Raw phone number as extracted.")
    country_code: Optional[str] = Field(
        default=None, description="Country code (e.g., +91).")
    national_number: Optional[str] = Field(
        default=None, description="National number without country code.")
    e164_phone: Optional[str] = Field(
        default=None, description="E.164 formatted phone number.")
    location: Optional[str] = Field(
        default=None, description="Location such as city, state, or country.")
    linkedin: Optional[str] = Field(
        default=None, description="LinkedIn profile URL.")
    github: Optional[str] = Field(
        default=None, description="GitHub profile URL.")
    website: Optional[str] = Field(
        default=None, description="Personal website or portfolio URL.")


class ExperienceItem(BaseModel):
    company: Optional[str] = Field(
        default=None, description="Company or organization name.")
    title: Optional[str] = Field(
        default=None, description="Job title or role.")
    location: Optional[str] = Field(
        default=None, description="Job location if present.")
    start_date: Optional[str] = Field(
        default=None, description="Start date as written in the resume.")
    end_date: Optional[str] = Field(
        default=None, description="End date as written in the resume.")
    is_current: Optional[bool] = Field(
        default=None, description="True if this is the current role.")
    description_bullets: list[str] = Field(
        default_factory=list, description="Resume bullet points for the role.")
    technologies: list[str] = Field(
        default_factory=list, description="Technologies, tools, or methods mentioned for the role.")


class EducationItem(BaseModel):
    institution: Optional[str] = Field(
        default=None, description="School, college, or university name.")
    degree: Optional[str] = Field(
        default=None, description="Degree or qualification.")
    field_of_study: Optional[str] = Field(
        default=None, description="Branch, major, or specialization.")
    start_date: Optional[str] = Field(
        default=None, description="Start date if present.")
    end_date: Optional[str] = Field(
        default=None, description="End date if present.")
    grade: Optional[str] = Field(
        default=None, description="CGPA, percentage, or grade if present.")
    notes: list[str] = Field(
        default_factory=list, description="Additional notes such as honors or relevant coursework.")


class ProjectItem(BaseModel):
    name: Optional[str] = Field(default=None, description="Project name.")
    description: Optional[str] = Field(
        default=None, description="Short description of the project.")
    technologies: list[str] = Field(
        default_factory=list, description="Tools, libraries, frameworks, or languages used.")
    link: Optional[str] = Field(
        default=None, description="Project URL if present.")


class CertificationItem(BaseModel):
    name: Optional[str] = Field(
        default=None, description="Certification name.")
    issuer: Optional[str] = Field(
        default=None, description="Issuing organization.")
    date: Optional[str] = Field(default=None, description="Date if present.")
    credential_id: Optional[str] = Field(
        default=None, description="Credential ID if present.")
    link: Optional[str] = Field(
        default=None, description="Credential URL if present.")


class ResumeExtraction(BaseModel):
    full_name: Optional[str] = Field(
        default=None, description="Candidate full name.")
    headline: Optional[str] = Field(
        default=None, description="Professional headline.")
    contact: ContactInfo
    summary: Optional[str] = Field(
        default=None, description="Professional summary.")

    skills: list[str] = Field(default_factory=list,
                              description="All extracted skills.")
    programming_languages: list[str] = Field(
        default_factory=list, description="Programming languages (Python, Java, C, etc.).")
    spoken_languages: list[str] = Field(
        default_factory=list, description="Human languages (English, Hindi, etc.).")

    experience: list[ExperienceItem] = Field(default_factory=list)
    education: list[EducationItem] = Field(default_factory=list)
    projects: list[ProjectItem] = Field(default_factory=list)
    certifications: list[CertificationItem] = Field(default_factory=list)

    keywords: list[str] = Field(
        default_factory=list, description="Important keywords for search/indexing.")

In [ ]:
# ============================================================
# Cell 4 — PDF text extraction helpers
# Extracts:
# - layout-aware text
# - embedded link annotations
# - visible plain-text URLs
# - canonical https:// normalization
# - labeled URL manifest for downstream parsing
# ============================================================
from phonenumbers import PhoneNumberFormat
import phonenumbers
import re
import json
from urllib.parse import urlparse
import fitz
import pymupdf4llm

URL_TRAILING_PUNCT = ".,;:!?)]}>'\""


def normalize_phone(raw_phone: str | None, default_region: str = "IN") -> dict:
    if not raw_phone:
        return {
            "phone_raw": None,
            "country_code": None,
            "national_number": None,
            "e164_phone": None,
        }

    try:
        parsed = phonenumbers.parse(raw_phone, default_region)
        if not phonenumbers.is_valid_number(parsed):
            return {
                "phone_raw": raw_phone,
                "country_code": None,
                "national_number": None,
                "e164_phone": None,
            }

        return {
            "phone_raw": raw_phone,
            "country_code": f"+{parsed.country_code}",
            "national_number": str(parsed.national_number),
            "e164_phone": phonenumbers.format_number(parsed, PhoneNumberFormat.E164),
        }
    except Exception:
        return {
            "phone_raw": raw_phone,
            "country_code": None,
            "national_number": None,
            "e164_phone": None,
        }


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def strip_trailing_punctuation(url: str) -> str:
    url = url.strip()
    while url and url[-1] in URL_TRAILING_PUNCT:
        url = url[:-1]
    return url


def normalize_url(url: str | None) -> str | None:
    if not url:
        return None

    url = strip_trailing_punctuation(url.strip())

    # remove wrapping angle brackets if present
    if url.startswith("<") and url.endswith(">"):
        url = url[1:-1].strip()

    # ignore mailto for this pipeline
    if url.lower().startswith("mailto:"):
        return None

    parsed = urlparse(url)

    # already canonical
    if parsed.scheme in {"http", "https"} and parsed.netloc:
        return url

    # scheme-less URL or bare domain/path
    if re.match(r"^(www\.)?([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}(/.*)?$", url):
        return "https://" + url

    return None


def extract_urls_from_text(text: str) -> list[str]:
    """
    Captures:
    - https://...
    - http://...
    - www....
    - bare domains with optional paths, e.g. github.com/user/repo
    """
    text = text or ""
    pattern = re.compile(
        r"""(?ix)
        (?<!@)
        \b(
            https?://[^\s<>()\[\]{}"']+
            |
            www\.[^\s<>()\[\]{}"']+
            |
            (?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}(?:/[^\s<>()\[\]{}"']+)?
        )
        """
    )

    found = []
    for m in pattern.finditer(text):
        url = normalize_url(m.group(1))
        if url:
            found.append(url)

    seen = set()
    out = []
    for url in found:
        if url not in seen:
            seen.add(url)
            out.append(url)
    return out


def extract_embedded_urls(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    urls = []

    for page in doc:
        for link in page.get_links():
            uri = normalize_url(link.get("uri"))
            if uri:
                urls.append(uri)

    seen = set()
    out = []
    for url in urls:
        if url not in seen:
            seen.add(url)
            out.append(url)
    return out


def classify_url(url: str) -> str:
    u = url.lower()
    if "linkedin.com" in u:
        return "contact.linkedin"
    if "github.com" in u:
        return "contact.github"
    if "twitter.com" in u or "x.com" in u:
        return "contact.twitter"
    if "instagram.com" in u:
        return "contact.instagram"
    if "leetcode.com" in u:
        return "contact.leetcode"
    if "kaggle.com" in u:
        return "contact.kaggle"
    return "other"


def build_url_manifest(urls: list[str]) -> str:
    """
    Produces a machine-readable manifest that Gemini can use.
    """
    manifest = []
    for url in urls:
        manifest.append({
            "kind": classify_url(url),
            "url": url,
            "domain": urlparse(url).netloc.lower() if urlparse(url).netloc else "",
        })
    return json.dumps(manifest, ensure_ascii=False, indent=2)


def extract_pdf_text(pdf_path: str) -> str:
    # 1) layout-aware extraction
    try:
        text = pymupdf4llm.to_markdown(pdf_path, header=False, footer=False)
        text = normalize_text(text) if text and text.strip() else ""
    except Exception as e:
        print(
            f"[extract_pdf_text] pymupdf4llm failed, falling back to PyMuPDF: {e}")
        text = ""

    # 2) raw text fallback
    if not text:
        doc = fitz.open(pdf_path)
        parts = []
        for page in doc:
            parts.append(page.get_text("text", sort=True))
        text = normalize_text("\n\n".join(parts))

    # 3) collect URLs from both annotations and visible text
    embedded_urls = extract_embedded_urls(pdf_path)
    visible_urls = extract_urls_from_text(text)

    all_urls = []
    seen = set()
    for url in embedded_urls + visible_urls:
        url = normalize_url(url)
        if url and url not in seen:
            seen.add(url)
            all_urls.append(url)

    # 4) append a strict manifest for the LLM
    if all_urls:
        text += "\n\n[EXTRACTED_URLS_JSON]\n"
        text += build_url_manifest(all_urls)
        text += "\n[/EXTRACTED_URLS_JSON]"

    return text

In [ ]:
# ============================================================
# Cell 5 — Gemini extraction and repair functions
# One normal extraction request.
# One repair request if validation fails.
# ============================================================
def build_extraction_prompt(resume_text: str) -> str:
    return f"""
You are a resume information extraction engine.

Return ONLY valid JSON matching the provided schema.
Do not add markdown, prose, code fences, or commentary.
Do not invent facts.
Use null where a value is unavailable.

Important rules:
- Put programming languages like Python, Java, C, C++, JavaScript in programming_languages.
- Put spoken human languages like English, Hindi, Marathi in spoken_languages.
- Put all other tools, libraries, frameworks, and technologies in skills.
- For contact.phone_raw, preserve the original phone text if present.
- Do not guess missing URLs.
- If a URL is visible in the resume or present in the extracted URL manifest, include it.
- Prefer the extracted URL manifest for linkedin/github/project links when possible.
- Keep dates as strings exactly as written when possible.

Resume text:
<<<BEGIN_RESUME_TEXT
{resume_text}
END_RESUME_TEXT>>>
""".strip()


def build_repair_prompt(resume_text: str, draft_json: str, validation_error: str) -> str:
    return f"""
You produced JSON that failed schema validation.

Fix it and return ONLY valid JSON matching the schema.
Do not add markdown, prose, or code fences.
Do not invent facts.
Use null where unknown.

Validation error:
{validation_error}

Previous JSON:
{draft_json}

Resume text:
<<<BEGIN_RESUME_TEXT
{resume_text}
END_RESUME_TEXT>>>
""".strip()


def _strip_code_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text, flags=re.IGNORECASE)
    return text.strip()


def gemini_generate_resume_json(prompt: str) -> str:
    import time

    MAX_RETRIES = 5
    BASE_WAIT_SECONDS = 8  # exponential backoff base

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json",
                    "response_json_schema": ResumeExtraction.model_json_schema(),
                },
            )

            result = _strip_code_fences(response.text or "")

            if not result:
                raise RuntimeError("Empty response received from Gemini.")

            return result

        except Exception as e:
            last_error = e
            error_text = str(e).lower()

            retryable_errors = [
                "503",
                "service unavailable",
                "resource exhausted",
                "quota",
                "rate limit",
                "429",
                "deadline exceeded",
                "timeout",
                "temporarily unavailable",
                "internal error",
                "500",
            ]

            should_retry = any(err in error_text for err in retryable_errors)

            if not should_retry:
                raise e

            if attempt < MAX_RETRIES:
                wait_time = BASE_WAIT_SECONDS * (2 ** (attempt - 1))

                print(
                    f"[Gemini Retry] Attempt {attempt}/{MAX_RETRIES} failed: {e}"
                )
                print(
                    f"[Gemini Retry] Waiting {wait_time} seconds before retry..."
                )

                time.sleep(wait_time)
            else:
                print(
                    f"[Gemini Retry] Final attempt failed after {MAX_RETRIES} retries."
                )

    raise RuntimeError(
        f"Gemini request failed after {MAX_RETRIES} retries. Last error: {last_error}"
    )


def parse_resume_json(candidate_json: str) -> ResumeExtraction:
    candidate_json = _strip_code_fences(candidate_json)
    return ResumeExtraction.model_validate_json(candidate_json)

In [ ]:
# ============================================================
# Cell 6 — LangGraph state and nodes
# ============================================================
class ResumeState(TypedDict, total=False):
    pdf_path: str
    raw_text: str
    draft_json: str
    validation_error: str
    extracted: dict
    output_path: str
    next_agent_input: str


def node_extract_text(state: ResumeState) -> dict:
    pdf_path = state["pdf_path"]
    raw_text = extract_pdf_text(pdf_path)
    return {"raw_text": raw_text}


def node_parse_resume(state: ResumeState) -> dict:
    raw_text = state["raw_text"]
    prompt = build_extraction_prompt(raw_text)
    draft_json = gemini_generate_resume_json(prompt)

    try:
        parsed = parse_resume_json(draft_json)
        parsed_dict = parsed.model_dump(mode="json")

        # Phone normalization
        contact = parsed_dict.get("contact", {}) or {}
        phone_raw = contact.get("phone_raw") or contact.get("phone")
        phone_data = normalize_phone(phone_raw)
        contact.update(phone_data)

        # URL normalization for contact fields
        for field in ("linkedin", "github", "website"):
            contact[field] = normalize_url(contact.get(field))

        parsed_dict["contact"] = contact

        return {
            "draft_json": draft_json,
            "validation_error": "",
            "extracted": parsed_dict,
        }

    except ValidationError as e:
        return {
            "draft_json": draft_json,
            "validation_error": str(e),
        }


def node_repair_resume(state: ResumeState) -> dict:
    raw_text = state["raw_text"]
    draft_json = state.get("draft_json", "")
    validation_error = state.get("validation_error", "")

    repair_prompt = build_repair_prompt(raw_text, draft_json, validation_error)
    repaired_json = gemini_generate_resume_json(repair_prompt)
    parsed = parse_resume_json(repaired_json)
    parsed_dict = parsed.model_dump(mode="json")

    # Phone normalization
    contact = parsed_dict.get("contact", {}) or {}
    phone_raw = contact.get("phone_raw") or contact.get("phone")
    phone_data = normalize_phone(phone_raw)
    contact.update(phone_data)

    # URL normalization for contact fields
    for field in ("linkedin", "github", "website"):
        contact[field] = normalize_url(contact.get(field))

    parsed_dict["contact"] = contact

    return {
        "draft_json": repaired_json,
        "validation_error": "",
        "extracted": parsed_dict,
    }


def node_save_result(state: ResumeState) -> dict:
    extracted = state["extracted"]

    pdf_path = Path(state["pdf_path"])
    output_path = pdf_path.with_suffix(".resume_extracted.json")

    output_path.write_text(
        json.dumps(extracted, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    next_agent_input = json.dumps(
        extracted, ensure_ascii=False, separators=(",", ":"))

    return {
        "output_path": str(output_path),
        "next_agent_input": next_agent_input,
        "extracted": extracted,   # <<< IMPORTANT FIX
    }

In [ ]:
# ============================================================
# Cell 7 — Build the graph
# parse -> repair only when validation failed
# ============================================================
def route_after_parse(state: ResumeState) -> str:
    if state.get("validation_error"):
        return "repair"
    return "save"


graph = StateGraph(ResumeState)

graph.add_node("extract_text", node_extract_text)
graph.add_node("parse_resume", node_parse_resume)
graph.add_node("repair_resume", node_repair_resume)
graph.add_node("save_result", node_save_result)

graph.add_edge(START, "extract_text")
graph.add_edge("extract_text", "parse_resume")
graph.add_conditional_edges(
    "parse_resume",
    route_after_parse,
    {
        "repair": "repair_resume",
        "save": "save_result",
    },
)
graph.add_edge("repair_resume", "save_result")
graph.add_edge("save_result", END)

app = graph.compile()

print("Graph compiled.")

Graph compiled.


In [ ]:
# ============================================================
# Cell 8 — Upload a resume PDF and run the pipeline
# ============================================================
# uploaded = files.upload()
# if not uploaded:
#     raise RuntimeError("No file uploaded.")

pdf_path = "/content/Tanush Tambe ML Resume.pdf"
print("Uploaded:", pdf_path)

result = app.invoke({"pdf_path": pdf_path})

print("\nSaved JSON path:", result["output_path"])
print("\nNext-agent input (compact JSON):")
print(result["next_agent_input"][:4000])  # print a safe preview

Uploaded: /content/Tanush Tambe ML Resume.pdf
=== Document parser messages ===
Using Tesseract for OCR processing.

[Gemini Retry] Attempt 1/5 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[Gemini Retry] Waiting 8 seconds before retry...
[Gemini Retry] Attempt 2/5 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[Gemini Retry] Waiting 16 seconds before retry...

Saved JSON path: /content/Tanush Tambe ML Resume.resume_extracted.json

Next-agent input (compact JSON):
{"full_name":"Tanush Sudheer Tambe","headline":"Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor 

In [ ]:
# ============================================================
# Cell 9 — Load the parsed result as a Pydantic object
# Useful if you want to continue downstream in the notebook.
# ============================================================
parsed_resume = ResumeExtraction.model_validate(result["extracted"])
print(parsed_resume.model_dump_json(indent=2))

{
  "full_name": "Tanush Sudheer Tambe",
  "headline": "Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor with a strong academic record from IIT Madras and Mumbai University.",
  "contact": {
    "email": "tambetanush@gmail.com",
    "phone_raw": "+91-7058915474",
    "country_code": "+91",
    "national_number": "7058915474",
    "e164_phone": "+917058915474",
    "location": "Karjat, Raigad, Maharashtra, India",
    "linkedin": "https://linkedin.com/in/tanush-tambe",
    "github": "https://github.com/tambetanush",
    "website": null
  },
  "summary": "Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor with a strong academic record from IIT Madras and Mumbai University.",
  "skills": [
    "Pandas",
    "NumPy",
    "Scikit-Learn",
    "XGBoost",


In [ ]:
# ============================================================
# Cell 9 — Evaluation Dataset Generator (5 Dummy Resume PDFs)
# Creates:
#   /content/test_resumes/
#       resume_1.pdf ... resume_5.pdf
#   /content/test_resumes/ground_truth.json
#
# Requires:
#   !pip install reportlab
# ============================================================

!pip -q install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.1 MB/s eta 0:00:00


In [ ]:
# from pathlib import Path
# import json
# from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
# from reportlab.lib.styles import getSampleStyleSheet

# BASE_DIR = Path("/content/test_resumes")
# BASE_DIR.mkdir(parents=True, exist_ok=True)

# styles = getSampleStyleSheet()


# def create_pdf(filepath: Path, lines: list[str]):
#     doc = SimpleDocTemplate(str(filepath))
#     story = []

#     for line in lines:
#         story.append(Paragraph(line.replace("\n", "<br/>"), styles["Normal"]))
#         story.append(Spacer(1, 8))

#     doc.build(story)


# test_cases = [
#     # ========================================================
#     # Resume 1 — Strong DS/ML profile
#     # ========================================================
#     {
#         "id": "resume_1",
#         "content": [
#             "Tanush Sudheer Tambe",
#             "Data Science and Machine Learning Practitioner",
#             "Email: tambetanush@gmail.com",
#             "Phone: +91-7058915474",
#             "Location: Karjat, Raigad, Maharashtra, India",
#             "LinkedIn: linkedin.com/in/tanush-tambe",
#             "GitHub: github.com/tambetanush",
#             "Summary: Applied ML practitioner with strong experience in ranking systems, IoT pipelines, and backend systems.",
#             "Programming Languages: Python, Java, C, JavaScript",
#             "Skills: Pandas, NumPy, Scikit-Learn, XGBoost, LightGBM, Docker, Flask, FastAPI, Redis",
#             "Languages: English, Hindi",
#             "Education: IIT Madras — BS Data Science and Applications — CGPA 9.58",
#             "Education: Konkan Gyanpeeth COE — BE Computer Engineering — CGPA 9.90",
#             "Project: GreenSense Crop Recommendation using ESP32, Flask, XGBoost",
#             "Project Link: github.com/tambetanush/GreenSense",
#             "Experience: Research Intern at AI Lab from Jan 2024 to May 2024",
#         ],
#         "ground_truth": {
#             "full_name": "Tanush Sudheer Tambe",
#             "email": "tambetanush@gmail.com",
#             "phone": "7058915474",
#             "country_code": "+91",
#             "linkedin": "https://linkedin.com/in/tanush-tambe",
#             "github": "https://github.com/tambetanush",
#             "programming_languages": ["Python", "Java", "C", "JavaScript"],
#             "spoken_languages": ["English", "Hindi"],
#             "skills": ["Pandas", "NumPy", "Scikit-Learn", "XGBoost", "LightGBM", "Docker", "Flask", "FastAPI", "Redis"]
#         }
#     },

#     # ========================================================
#     # Resume 2 — Backend with internship
#     # ========================================================
#     {
#         "id": "resume_2",
#         "content": [
#             "Aditi Sharma",
#             "Backend Developer",
#             "Email: aditi.sharma@gmail.com",
#             "Phone: 9876543210",
#             "Location: Pune, India",
#             "Website: www.aditisharma.dev",
#             "Summary: Backend engineer focused on APIs, distributed jobs, and async systems.",
#             "Programming Languages: Python",
#             "Skills: FastAPI, PostgreSQL, Redis, Celery, Docker, SQLAlchemy",
#             "Languages: English, Marathi",
#             "Experience: Software Engineer Intern at ABC Tech from June 2023 to Dec 2023",
#             "Experience: Backend Developer at XYZ Labs from Jan 2024 to Present",
#             "Project: API Gateway System with rate limiting and Celery workers",
#             "Certification: Redis for Developers",
#         ],
#         "ground_truth": {
#             "full_name": "Aditi Sharma",
#             "email": "aditi.sharma@gmail.com",
#             "phone": "9876543210",
#             "country_code": "+91",
#             "programming_languages": ["Python"],
#             "spoken_languages": ["English", "Marathi"],
#             "skills": ["FastAPI", "PostgreSQL", "Redis", "Celery", "Docker", "SQLAlchemy"]
#         }
#     },

#     # ========================================================
#     # Resume 3 — ML Engineer with certification
#     # ========================================================
#     {
#         "id": "resume_3",
#         "content": [
#             "Rahul Verma",
#             "Machine Learning Engineer",
#             "Email: rahulv@gmail.com",
#             "Phone: +91 9988776655",
#             "GitHub: github.com/rahulverma",
#             "Summary: ML engineer focused on model deployment and feature pipelines.",
#             "Programming Languages: Python",
#             "Skills: Scikit-Learn, LightGBM, AWS, Kubernetes, MLflow",
#             "Languages: English",
#             "Certification: AWS Cloud Practitioner",
#             "Certification Link: https://aws.amazon.com/certification/",
#             "Project: Fraud Detection Pipeline",
#         ],
#         "ground_truth": {
#             "full_name": "Rahul Verma",
#             "email": "rahulv@gmail.com",
#             "phone": "9988776655",
#             "country_code": "+91",
#             "github": "https://github.com/rahulverma",
#             "programming_languages": ["Python"],
#             "spoken_languages": ["English"],
#             "skills": ["Scikit-Learn", "LightGBM", "AWS", "Kubernetes", "MLflow"]
#         }
#     },

#     # ========================================================
#     # Resume 4 — Frontend
#     # ========================================================
#     {
#         "id": "resume_4",
#         "content": [
#             "Sneha Patil",
#             "Frontend Developer",
#             "Email: sneha@gmail.com",
#             "Phone: (+91) 91234 56780",
#             "LinkedIn: linkedin.com/in/snehapatil",
#             "Programming Languages: JavaScript",
#             "Skills: Vue.js, HTML, CSS, Bootstrap, TailwindCSS",
#             "Languages: English, Hindi, Marathi",
#             "Project: Student Portal Dashboard",
#             "Experience: Frontend Intern at WebLabs",
#         ],
#         "ground_truth": {
#             "full_name": "Sneha Patil",
#             "email": "sneha@gmail.com",
#             "phone": "9123456780",
#             "country_code": "+91",
#             "linkedin": "https://linkedin.com/in/snehapatil",
#             "programming_languages": ["JavaScript"],
#             "spoken_languages": ["English", "Hindi", "Marathi"],
#             "skills": ["Vue.js", "HTML", "CSS", "Bootstrap", "TailwindCSS"]
#         }
#     },

#     # ========================================================
#     # Resume 5 — IoT profile
#     # ========================================================
#     {
#         "id": "resume_5",
#         "content": [
#             "Vikram Joshi",
#             "IoT and Embedded Systems Engineer",
#             "Email: vikram@gmail.com",
#             "Phone: 9012345678",
#             "Programming Languages: C, Python",
#             "Skills: ESP32, Raspberry Pi Pico, MQTT, Flask, Sensors, Arduino",
#             "Languages: English",
#             "Project: Smart Irrigation Monitoring System",
#             "Project: Cattle Health Monitoring System",
#         ],
#         "ground_truth": {
#             "full_name": "Vikram Joshi",
#             "email": "vikram@gmail.com",
#             "phone": "9012345678",
#             "country_code": "+91",
#             "programming_languages": ["C", "Python"],
#             "spoken_languages": ["English"],
#             "skills": ["ESP32", "Raspberry Pi Pico", "MQTT", "Flask", "Sensors", "Arduino"]
#         }
#     },

#     # ========================================================
#     # Resume 6 — Missing contact links edge case
#     # ========================================================
#     {
#         "id": "resume_6",
#         "content": [
#             "Neha Kulkarni",
#             "Data Analyst",
#             "Email: neha.k@gmail.com",
#             "Phone: 8899776655",
#             "Programming Languages: Python, SQL",
#             "Skills: Power BI, Tableau, Excel, Pandas",
#             "Languages: English, Hindi",
#             "Project: Sales Forecast Dashboard",
#         ],
#         "ground_truth": {
#             "full_name": "Neha Kulkarni",
#             "email": "neha.k@gmail.com",
#             "phone": "8899776655",
#             "country_code": "+91",
#             "programming_languages": ["Python", "SQL"],
#             "spoken_languages": ["English", "Hindi"],
#             "skills": ["Power BI", "Tableau", "Excel", "Pandas"]
#         }
#     },

#     # ========================================================
#     # Resume 7 — Portfolio + certification links
#     # ========================================================
#     {
#         "id": "resume_7",
#         "content": [
#             "Arjun Mehta",
#             "Full Stack Developer",
#             "Email: arjun@gmail.com",
#             "Phone: +91 9090909090",
#             "Portfolio: arjunmehta.dev",
#             "GitHub: github.com/arjunmehta",
#             "Programming Languages: JavaScript, TypeScript",
#             "Skills: React, Node.js, Express, MongoDB, Docker",
#             "Languages: English",
#             "Certification: MongoDB Associate Developer",
#             "Certification Link: mongodb.com/certification",
#         ],
#         "ground_truth": {
#             "full_name": "Arjun Mehta",
#             "email": "arjun@gmail.com",
#             "phone": "9090909090",
#             "country_code": "+91",
#             "github": "https://github.com/arjunmehta",
#             "programming_languages": ["JavaScript", "TypeScript"],
#             "spoken_languages": ["English"],
#             "skills": ["React", "Node.js", "Express", "MongoDB", "Docker"]
#         }
#     },

#     # ========================================================
#     # Resume 8 — No spoken language section edge case
#     # ========================================================
#     {
#         "id": "resume_8",
#         "content": [
#             "Priya Nair",
#             "Cloud Engineer",
#             "Email: priya@gmail.com",
#             "Phone: 8080808080",
#             "LinkedIn: linkedin.com/in/priyanair",
#             "Programming Languages: Python, Bash",
#             "Skills: AWS, Terraform, Kubernetes, Docker, Jenkins",
#             "Experience: DevOps Engineer at CloudOps",
#             "Certification: AWS Solutions Architect",
#         ],
#         "ground_truth": {
#             "full_name": "Priya Nair",
#             "email": "priya@gmail.com",
#             "phone": "8080808080",
#             "country_code": "+91",
#             "linkedin": "https://linkedin.com/in/priyanair",
#             "programming_languages": ["Python", "Bash"],
#             "spoken_languages": [],
#             "skills": ["AWS", "Terraform", "Kubernetes", "Docker", "Jenkins"]
#         }
#     }
# ]


# ground_truth_all = {}

# for case in test_cases:
#     pdf_path = BASE_DIR / f"{case['id']}.pdf"
#     create_pdf(pdf_path, case["content"])
#     ground_truth_all[case["id"]] = case["ground_truth"]

# gt_path = BASE_DIR / "ground_truth.json"
# gt_path.write_text(
#     json.dumps(ground_truth_all, indent=2),
#     encoding="utf-8"
# )

# print("Created PDFs in:", BASE_DIR)
# print("Ground truth file:", gt_path)
# print(f"Total test resumes created: {len(test_cases)}")

In [ ]:
# # ============================================================
# # Cell 10 — Run Your Existing Resume Extractor on Test PDFs
# # Uses your existing:
# #   app.invoke(...)
# # ============================================================

# predictions = {}

# for pdf_file in sorted(BASE_DIR.glob("resume_*.pdf")):
#     print(f"Running extractor on: {pdf_file.name}")

#     result = app.invoke({"pdf_path": str(pdf_file)})

#     extracted_path = Path(result["output_path"])
#     extracted_json = json.loads(extracted_path.read_text(encoding="utf-8"))

#     predictions[pdf_file.stem] = extracted_json

# pred_path = BASE_DIR / "predictions.json"
# pred_path.write_text(json.dumps(predictions, indent=2), encoding="utf-8")

# print("\nSaved predictions:", pred_path)

In [ ]:
# # ============================================================
# # Cell 11 — Evaluation Metrics
# #
# # Metrics:
# #   - exact match for scalar fields
# #   - precision / recall / F1 for list fields
# #   - hallucination count
# # ============================================================

# from collections import defaultdict


# def normalize_string(x):
#     if x is None:
#         return None
#     return str(x).strip().lower()


# def set_metrics(gt_list, pred_list):
#     gt = set(map(normalize_string, gt_list or []))
#     pred = set(map(normalize_string, pred_list or []))

#     tp = len(gt & pred)
#     fp = len(pred - gt)
#     fn = len(gt - pred)

#     precision = tp / (tp + fp) if (tp + fp) else 0
#     recall = tp / (tp + fn) if (tp + fn) else 0
#     f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

#     return {
#         "precision": round(precision, 4),
#         "recall": round(recall, 4),
#         "f1": round(f1, 4),
#         "hallucinations": fp
#     }


# def scalar_match(gt, pred):
#     return int(normalize_string(gt) == normalize_string(pred))


# evaluation_results = {}

# for rid, gt in ground_truth_all.items():
#     pred = predictions.get(rid, {})

#     contact = pred.get("contact", {})

#     result = {
#         "name_match": scalar_match(gt.get("full_name"), pred.get("full_name")),
#         "email_match": scalar_match(gt.get("email"), contact.get("email")),
#         "phone_match": scalar_match(gt.get("phone"), contact.get("national_number")),
#         "country_code_match": scalar_match(gt.get("country_code"), contact.get("country_code")),
#         "linkedin_match": scalar_match(gt.get("linkedin"), contact.get("linkedin")),
#         "github_match": scalar_match(gt.get("github"), contact.get("github")),
#         "programming_languages": set_metrics(
#             gt.get("programming_languages", []),
#             pred.get("programming_languages", [])
#         ),
#         "spoken_languages": set_metrics(
#             gt.get("spoken_languages", []),
#             pred.get("spoken_languages", [])
#         ),
#         "skills": set_metrics(
#             gt.get("skills", []),
#             pred.get("skills", [])
#         )
#     }

#     evaluation_results[rid] = result

# eval_path = BASE_DIR / "evaluation_results.json"
# eval_path.write_text(json.dumps(evaluation_results, indent=2), encoding="utf-8")

# print(json.dumps(evaluation_results, indent=2))
# print("\nSaved:", eval_path)

In [ ]:
# # ============================================================
# # Cell 12 — Final Summary Table
# # ============================================================

# summary = defaultdict(list)

# for rid, res in evaluation_results.items():
#     summary["name_match"].append(res["name_match"])
#     summary["email_match"].append(res["email_match"])
#     summary["phone_match"].append(res["phone_match"])
#     summary["country_code_match"].append(res["country_code_match"])

#     summary["skills_f1"].append(res["skills"]["f1"])
#     summary["prog_lang_f1"].append(res["programming_languages"]["f1"])
#     summary["spoken_lang_f1"].append(res["spoken_languages"]["f1"])


# final_summary = {
#     k: round(sum(v) / len(v), 4)
#     for k, v in summary.items()
# }

# print("========== FINAL EVALUATION SUMMARY ==========")
# print(json.dumps(final_summary, indent=2))

In [ ]:
!pip -q install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.7 MB/s eta 0:00:00


In [ ]:
from supabase import create_client

SUPABASE_URL = "https://lotcwutyfwnsdscmzhcl.supabase.co"
SUPABASE_KEY = userdata.get('SUPABASE_KEY')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
def insert_full_resume(data: dict):
    res = supabase.table("resumes").insert({
        "full_name": data.get("full_name"),
        "headline": data.get("headline"),
        "summary": data.get("summary"),
    }).execute()

    resume_id = res.data[0]["id"]

    contact = data.get("contact", {})
    supabase.table("contacts").insert({
        "resume_id": resume_id,
        **contact
    }).execute()

    for s in data.get("skills", []):
        supabase.table("skills").insert({
            "resume_id": resume_id,
            "skill": s
        }).execute()

    for s in data.get("programming_languages", []):
        supabase.table("programming_languages").insert({
            "resume_id": resume_id,
            "language": s
        }).execute()

    for s in data.get("spoken_languages", []):
        supabase.table("spoken_languages").insert({
            "resume_id": resume_id,
            "language": s
        }).execute()

    for k in data.get("keywords", []):
        supabase.table("keywords").insert({
            "resume_id": resume_id,
            "keyword": k
        }).execute()

    for exp in data.get("experience", []):
        exp_res = supabase.table("experiences").insert({
            "resume_id": resume_id,
            "company": exp.get("company"),
            "title": exp.get("title"),
            "location": exp.get("location"),
            "start_date": exp.get("start_date"),
            "end_date": exp.get("end_date"),
            "is_current": exp.get("is_current"),
        }).execute()

        exp_id = exp_res.data[0]["id"]

        for b in exp.get("description_bullets", []):
            supabase.table("experience_bullets").insert({
                "experience_id": exp_id,
                "bullet": b
            }).execute()

        for t in exp.get("technologies", []):
            supabase.table("experience_technologies").insert({
                "experience_id": exp_id,
                "tech": t
            }).execute()

    for edu in data.get("education", []):
        edu_res = supabase.table("education").insert({
            "resume_id": resume_id,
            **{k: edu.get(k) for k in ["institution", "degree", "field_of_study", "start_date", "end_date", "grade"]}
        }).execute()

        edu_id = edu_res.data[0]["id"]

        for n in edu.get("notes", []):
            supabase.table("education_notes").insert({
                "education_id": edu_id,
                "note": n
            }).execute()

    for proj in data.get("projects", []):
        proj_res = supabase.table("projects").insert({
            "resume_id": resume_id,
            "name": proj.get("name"),
            "description": proj.get("description"),
            "link": proj.get("link"),
        }).execute()

        proj_id = proj_res.data[0]["id"]

        for t in proj.get("technologies", []):
            supabase.table("project_technologies").insert({
                "project_id": proj_id,
                "tech": t
            }).execute()

    for cert in data.get("certifications", []):
        supabase.table("certifications").insert({
            "resume_id": resume_id,
            **cert
        }).execute()

    return resume_id

In [ ]:
resume_id = insert_full_resume(result["extracted"])
print("Inserted resume_id:", resume_id)

Inserted resume_id: 2224abd7-735e-4084-add7-be1cb2c0f046


In [ ]:
def fetch_full_resume(resume_id: str) -> dict:
    # main
    resume = supabase.table("resumes").select(
        "*").eq("id", resume_id).execute().data[0]

    # contact
    contact = supabase.table("contacts").select(
        "*").eq("resume_id", resume_id).execute().data
    contact = contact[0] if contact else {}

    # skills
    skills = [x["skill"] for x in supabase.table("skills").select(
        "*").eq("resume_id", resume_id).execute().data]

    prog_langs = [x["language"] for x in supabase.table(
        "programming_languages").select("*").eq("resume_id", resume_id).execute().data]

    spoken_langs = [x["language"] for x in supabase.table(
        "spoken_languages").select("*").eq("resume_id", resume_id).execute().data]

    keywords = [x["keyword"] for x in supabase.table("keywords").select(
        "*").eq("resume_id", resume_id).execute().data]

    # experience
    experiences = []
    exp_rows = supabase.table("experiences").select(
        "*").eq("resume_id", resume_id).execute().data

    for exp in exp_rows:
        exp_id = exp["id"]

        bullets = [x["bullet"] for x in supabase.table("experience_bullets").select(
            "*").eq("experience_id", exp_id).execute().data]

        techs = [x["tech"] for x in supabase.table("experience_technologies").select(
            "*").eq("experience_id", exp_id).execute().data]

        exp["description_bullets"] = bullets
        exp["technologies"] = techs

        experiences.append(exp)

    # education
    education = []
    edu_rows = supabase.table("education").select(
        "*").eq("resume_id", resume_id).execute().data

    for edu in edu_rows:
        edu_id = edu["id"]
        notes = [x["note"] for x in supabase.table("education_notes").select(
            "*").eq("education_id", edu_id).execute().data]
        edu["notes"] = notes
        education.append(edu)

    # projects
    projects = []
    proj_rows = supabase.table("projects").select(
        "*").eq("resume_id", resume_id).execute().data

    for proj in proj_rows:
        proj_id = proj["id"]
        techs = [x["tech"] for x in supabase.table("project_technologies").select(
            "*").eq("project_id", proj_id).execute().data]
        proj["technologies"] = techs
        projects.append(proj)

    # certifications
    certifications = supabase.table("certifications").select(
        "*").eq("resume_id", resume_id).execute().data

    return {
        "full_name": resume.get("full_name"),
        "headline": resume.get("headline"),
        "summary": resume.get("summary"),
        "contact": contact,
        "skills": skills,
        "programming_languages": prog_langs,
        "spoken_languages": spoken_langs,
        "experience": experiences,
        "education": education,
        "projects": projects,
        "certifications": certifications,
        "keywords": keywords,
    }

In [ ]:
fetched = fetch_full_resume(resume_id)

print(json.dumps(fetched, indent=2, ensure_ascii=False))

{
  "full_name": "Tanush Sudheer Tambe",
  "headline": "Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor with a strong academic record from IIT Madras and Mumbai University.",
  "summary": "Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor with a strong academic record from IIT Madras and Mumbai University.",
  "contact": {
    "id": "c3d1891c-1116-408f-9c7d-0bf719dc09bb",
    "resume_id": "2224abd7-735e-4084-add7-be1cb2c0f046",
    "email": "tambetanush@gmail.com",
    "phone_raw": "+91-7058915474",
    "country_code": "+91",
    "national_number": "7058915474",
    "e164_phone": "+917058915474",
    "location": "Karjat, Raigad, Maharashtra, India",
    "linkedin": "https://linkedin.com/in/tanush-tambe",
    "github": "https://github.com/tambetan

In [ ]:
validated = ResumeExtraction.model_validate(fetched)
print(validated.model_dump_json(indent=2))

{
  "full_name": "Tanush Sudheer Tambe",
  "headline": "Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor with a strong academic record from IIT Madras and Mumbai University.",
  "contact": {
    "email": "tambetanush@gmail.com",
    "phone_raw": "+91-7058915474",
    "country_code": "+91",
    "national_number": "7058915474",
    "e164_phone": "+917058915474",
    "location": "Karjat, Raigad, Maharashtra, India",
    "linkedin": "https://linkedin.com/in/tanush-tambe",
    "github": "https://github.com/tambetanush",
    "website": null
  },
  "summary": "Data Science and Machine Learning practitioner with strong experience in applied machine learning, IoTdriven data pipelines, and backend systems. Top-1% Kaggle competitor with a strong academic record from IIT Madras and Mumbai University.",
  "skills": [
    "Pandas",
    "NumPy",
    "Scikit-Learn",
    "XGBoost",


In [ ]:
!pip -q install requests pydantic numpy

In [ ]:
import os
import json
import re
import requests
import numpy as np
from typing import Optional, List
from pydantic import BaseModel, Field

# ====== SET THESE ======
ADZUNA_APP_ID = ""
ADZUNA_APP_KEY = ""

JINA_API_KEY = ""  # https://jina.ai/embeddings

# Gemini function already exists in your notebook:
# gemini_generate_resume_json(prompt: str) -> str

In [ ]:
class ScoreBreakdown(BaseModel):
    semantic: float
    skill_overlap: float
    experience: float
    education: float
    final: float


class JobExplanation(BaseModel):
    strengths: List[str]
    gaps: List[str]
    reasoning: str


class JobResult(BaseModel):
    job_id: str
    title: str
    company: Optional[str]
    location: Optional[str]
    apply_url: Optional[str]

    score: ScoreBreakdown
    explanation: JobExplanation


class JobSearchResponse(BaseModel):
    query_role: str
    user_location_preference: str
    total_jobs_fetched: int
    jobs: List[JobResult]

In [ ]:
def embed_batch(texts: List[str]) -> np.ndarray:
    url = "https://api.jina.ai/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {JINA_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": "jina-embeddings-v2-base-en",
        "input": texts,
    }

    r = requests.post(url, headers=headers, json=payload, timeout=30)
    r.raise_for_status()
    data = r.json()["data"]
    # preserve order
    return np.array([d["embedding"] for d in data], dtype=np.float32)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

In [ ]:
def build_resume_representation(resume: dict):
    skills = set(
        s.lower() for s in (
            (resume.get("skills") or []) +
            (resume.get("programming_languages") or [])
        )
    )

    exp_text = " ".join(
        " ".join(e.get("description_bullets", []))
        for e in (resume.get("experience") or [])
    )

    proj_text = " ".join(
        (p.get("description") or "")
        for p in (resume.get("projects") or [])
    )

    summary = resume.get("summary") or ""

    structured = {
        "skills": skills,
        "has_experience": len(resume.get("experience") or []) > 0,
        "has_education": len(resume.get("education") or []) > 0,
    }

    semantic_text = " ".join([summary, exp_text, proj_text]).strip()
    return structured, semantic_text

In [ ]:
def safe_json_parse(text: str):
    try:
        parsed = json.loads(text)

        # case 1: already correct
        if isinstance(parsed, list):
            return parsed

        # case 2: stringified JSON
        if isinstance(parsed, str):
            return json.loads(parsed)

        # case 3: single object → wrap as list
        if isinstance(parsed, dict):
            return [parsed]

    except Exception as e:
        print("[JSON PARSE ERROR]", e)

    return []


def fetch_jobs(query_role: str, where: str, results: int = 20):
    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "what": query_role,
        "where": where if where not in ["remote", "hybrid"] else "",
        "results_per_page": results,
        "content-type": "application/json",
    }

    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()

    jobs = []
    for j in data.get("results", []):
        jobs.append({
            "job_id": str(j.get("id")) if j.get("id") is not None else None,
            "title": j.get("title"),
            "company": (j.get("company") or {}).get("display_name"),
            "location": (j.get("location") or {}).get("display_name"),
            "description": j.get("description") or "",
            "apply_url": j.get("redirect_url"),
        })
    return jobs

In [ ]:
def location_filter(job_loc: str, user_pref: str) -> bool:
    if not job_loc:
        return True

    job_loc_l = job_loc.lower()
    pref = (user_pref or "").lower().strip()

    if pref == "remote":
        return "remote" in job_loc_l

    if pref == "hybrid":
        return "hybrid" in job_loc_l or "remote" in job_loc_l

    # city match
    return pref in job_loc_l

In [ ]:
def tokenize(text: str):
    return set(re.findall(r"[a-zA-Z0-9\+\#\.]+", text.lower()))


def score_jobs(resume: dict, jobs: list):
    structured, resume_text = build_resume_representation(resume)

    # Build texts
    job_texts = [
        (j["title"] + " " + j["description"]).strip()
        for j in jobs
    ]

    # Batch embeddings (1 call)
    all_texts = [resume_text] + job_texts
    embeds = embed_batch(all_texts)

    r_emb = embeds[0]
    j_embs = embeds[1:]

    results = []

    for i, job in enumerate(jobs):
        j_text = job_texts[i]
        j_emb = j_embs[i]

        semantic = cosine_sim(r_emb, j_emb)

        job_tokens = tokenize(j_text)
        overlap = len(structured["skills"] & job_tokens)
        skill_score = overlap / (len(job_tokens) + 1e-9)

        exp_score = 1.0 if structured["has_experience"] else 0.4
        edu_score = 1.0 if structured["has_education"] else 0.5

        final = (
            0.6 * semantic +
            0.25 * skill_score +
            0.1 * exp_score +
            0.05 * edu_score
        )

        results.append({
            **job,
            "_scores": {
                "semantic": semantic,
                "skill_overlap": skill_score,
                "experience": exp_score,
                "education": edu_score,
                "final": final,
            }
        })

    return results

In [ ]:
def build_bulk_explanation_prompt(resume: dict, jobs: list):
    compact_jobs = [
        {
            "job_id": j["job_id"],
            "title": j["title"],
            "company": j["company"],
            "description": j["description"][:1200],  # cap tokens
            "scores": j["_scores"],
        }
        for j in jobs
    ]

    return f"""
You are evaluating job fit.
Return ONLY a valid JSON array.
Do not wrap in string.
Do not escape JSON.
Do not return a single object.

Resume:
{resume}

Jobs:
{json.dumps(compact_jobs)}

Return JSON array:
[
  {{
    "job_id": "...",
    "strengths": [],
    "gaps": [],
    "reasoning": ""
  }}
]
"""

In [ ]:
def job_finder_agent(
    resume: dict,
    target_role: str,
    user_location_pref: str,
    top_k: int = 5
):
    # 1. fetch real jobs
    jobs = fetch_jobs(target_role, user_location_pref, results=25)

    # 2. location filter
    jobs = [j for j in jobs if location_filter(
        j.get("location"), user_location_pref)]

    if not jobs:
        return {
            "query_role": target_role,
            "user_location_preference": user_location_pref,
            "total_jobs_fetched": 0,
            "jobs": []
        }

    # 3. scoring (batched embeddings)
    scored = score_jobs(resume, jobs)

    # 4. select top-k candidates BEFORE LLM call
    scored = sorted(scored, key=lambda x: x["_scores"]["final"], reverse=True)[
        :max(top_k, 7)]

    # 5. single LLM call
    prompt = build_bulk_explanation_prompt(resume, scored)
    expl_json = gemini_generate_resume_json(prompt)

    parsed_explanations = safe_json_parse(expl_json)

    # filter invalid entries
    explanations = {
        e.get("job_id"): e
        for e in parsed_explanations
        if isinstance(e, dict) and "job_id" in e
    }

    # 6. build final output
    out_jobs = []
    for j in scored[:top_k]:
        s = j["_scores"]
        e = explanations.get(j["job_id"], {
            "strengths": [],
            "gaps": [],
            "reasoning": "No explanation generated."
        })

        out_jobs.append({
            "job_id": j["job_id"],
            "title": j["title"],
            "company": j["company"],
            "location": j["location"],
            "apply_url": j["apply_url"],
            "score": {
                "semantic": round(s["semantic"], 4),
                "skill_overlap": round(s["skill_overlap"], 4),
                "experience": s["experience"],
                "education": s["education"],
                "final": round(s["final"] * 100, 2),
            },
            "explanation": e
        })

    result = {
        "query_role": target_role,
        "user_location_preference": user_location_pref,
        "total_jobs_fetched": len(jobs),
        "jobs": out_jobs
    }

    return JobSearchResponse.model_validate(result).model_dump()

In [ ]:
job_output = job_finder_agent(
    resume=result["extracted"],   # from your resume agent
    target_role="Machine Learning Engineer",
    user_location_pref="Pune",  # or "Pune", "hybrid"
    top_k=5
)

print(json.dumps(job_output, indent=2))

{
  "query_role": "Machine Learning Engineer",
  "user_location_preference": "Pune",
  "total_jobs_fetched": 25,
  "jobs": [
    {
      "job_id": "5652053919",
      "title": "Data Scientist or Senior Machine Learning Engineer",
      "company": "Peak Hire Solutions",
      "location": "Pune, Maharashtra",
      "apply_url": "https://www.adzuna.in/details/5652053919?utm_medium=api&utm_source=6cf0cb2f",
      "score": {
        "semantic": 0.8456,
        "skill_overlap": 0.0182,
        "experience": 0.4,
        "education": 1.0,
        "final": 60.19
      },
      "explanation": {
        "strengths": [],
        "gaps": [],
        "reasoning": "No explanation generated."
      }
    },
    {
      "job_id": "5657222643",
      "title": "Data Scientist or Senior Machine Learning Engineer",
      "company": "Peak Hire Solutions",
      "location": "Pune, Maharashtra",
      "apply_url": "https://www.adzuna.in/details/5657222643?utm_medium=api&utm_source=6cf0cb2f",
      "score": {